## Majority Voting




In [2]:
#!pip install stats

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for stats: filename=stats-0.1.2a0-py3-none-any.whl size=24362 sha256=63761fbcfb5c82fa656a79fbfaae4b55d89097466692a3bb047495fc70a8873b
  Stored in directory: /root/.cache/pip/wheels/4d/48/cc/4eed8badca2795500d02975782bd130d12a79ba63b17f0185c
Successfully built stats


In [19]:
# import libraries
import numpy as np
import pandas as pd
import torch
#import transformers
from tqdm import tqdm
import random
import time
import stats
import datasets
import matplotlib.pyplot as plt

# sklearn
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn import linear_model
from sklearn import metrics
from sklearn.metrics import accuracy_score
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report,confusion_matrix
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import f1_score

# Hide warnings for alphas
from warnings import simplefilter
from sklearn.exceptions import ConvergenceWarning
simplefilter("ignore", category=ConvergenceWarning)

In [20]:
# set random seed
seed = torch.randint(0, 1000, (1,)).item()
seed # record for replication

978

## Political tweets dataset (Hugging Face)

In [4]:
# import the Hugging Face datasets library (pip install datasets)
from datasets import load_dataset

# download the political tweets dataset from the Hugging Face Hub
dataset = load_dataset("Jacobvs/PoliticalTweets")

# the dataset can contain several splits (e.g. train/test); take the first one
split_name = list(dataset.keys())[0]

# convert that split into a pandas DataFrame
df_raw = dataset[split_name].to_pandas()

# quick check of the data
df_raw.head(2)

README.md:   0%|          | 0.00/21.0 [00:00<?, ?B/s]

formatted_data.parquet: reconstructing file:   0%|          |  0.00B / 26.6MB            

formatted_data.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/190491 [00:00<?, ? examples/s]

,index,date,id,username,text,party,labels
0,0,2021-10-13 19:47:44,1448374915636383745,SenatorHassan,Happy th birthday to the @USNavy! The strength...,Democrat,1
1,1,2021-06-30 14:53:13,1410250073003462656,SenatorMenendez,The greatest generation's investment in infras...,Democrat,1


In [6]:
# remove duplicates
df_labelled = df_raw.drop_duplicates(subset=['text'], keep='first')

In [8]:
# check distribution
df_labelled.labels.value_counts()

,count
labels,
1,97808
0,92499


In [9]:
# Rename & set variable as int
df_labelled['label']=df_labelled['labels']
df_labelled['label'] = df_labelled['label'].astype('int')

/tmp/ipykernel_1552/3925489681.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_labelled['label']=df_labelled['labels']
/tmp/ipykernel_1552/3925489681.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_labelled['label'] = df_labelled['label'].astype('int')


In [10]:
# Subset
df=df_labelled[['text', 'label']]

In [11]:
# Convert Label to category
df['label'] = pd.Categorical(df.label)

/tmp/ipykernel_1552/1330544672.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['label'] = pd.Categorical(df.label)


In [12]:
# Convert labels to a list for checks
list_q1 = list(df_labelled['label'])
set(list_q1)

{0, 1}

In [13]:
# Create new lists for tokenizer and next steps
labels = df_labelled.label.to_list()
labels[50:55] #check

[1, 1, 1, 1, 1]

In [16]:
# create list of text
tweets = list(df_labelled['text'])
tweets[:10]

['Happy th birthday to the @USNavy! The strength, dedication, and skill of our Sailors including those at Portsmouth Naval Shipyard help keep this country safe, secure, and free. Today we recognize and celebrate their incredible service. #246NavyBirthday https://t.co/GuHEDMApke',
 "The greatest generation's investment in infrastructure made us the envy of the world. But now we've gone almost an entire lifetime without making any significant investments in the NEXT generation of American infrastructure. It's time for that to change. https://t.co/brikl79Kgp",
 'Thanks to @SenTedCruz and  @SenatorWarnock, the Infrastructure Investment and Jobs Act authorizes the Interstate East-West corridor across TX, LA, MS, AL, and GA. Working together to deliver real infrastructure!',
 '/ To get lasting change we cant just lock up those convicted of these crimes, but must also work to combat bias and  bigotry. The NO HATE Act would allow courts to implement alternative sentencingrequiring people to do

## Split in training and testing set

In [21]:
# convert list of labels into numpy array
labels = np.array(labels)
labels[0:10]

array([1, 1, 0, 1, 0, 1, 1, 0, 0, 1])

In [22]:
# Split data in training and testing sets
X_train, X_test, y_train, y_test = train_test_split(tweets, labels, test_size=0.20, random_state=seed) #stratify=q2_labels_array,

## Majority Voting Classifier

In [23]:
# define majority vote function
def majority_vote(y_train, y_test):
    pred_mode = stats.mode(y_train)
    pred_mode = pred_mode.repeat(y_test.shape[0])
    return pred_mode

In [24]:
# define predictions and truth
y_pred = majority_vote(y_train, y_test)
y_true = y_test

In [25]:
# print results
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

[[    0 18507]
 [    0 19555]]
              precision    recall  f1-score   support

           0       0.00      0.00      0.00     18507
           1       0.51      1.00      0.68     19555

    accuracy                           0.51     38062
   macro avg       0.26      0.50      0.34     38062
weighted avg       0.26      0.51      0.35     38062



/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [26]:
# AUC test (Goal: >0.7)
fpr, tpr, thresholds = metrics.roc_curve(y_true, y_pred, pos_label=1)
metrics.auc(fpr, tpr)

np.float64(0.5)